# MMR and Retrieval Metrics

You will diversify retrieval and evaluate retrieval quality with clear metrics.


In [ ]:
# DEPENDENCY: pip install -q openai numpy
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import numpy as np
from getpass import getpass
from dataclasses import dataclass, field
from openai import OpenAI

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"

In [ ]:
def get_embedding(text: str) -> np.ndarray:
    """Get embedding for a single text."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return np.array(response.data[0].embedding)

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_embeddings_batch(texts: list[str]) -> list[np.ndarray]:
    """Get embeddings for multiple texts efficiently."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [np.array(item.embedding) for item in response.data]

## Part 1 MMR

You will balance relevance and diversity to avoid repeated results.


In [ ]:
# Sample documents about Python (intentionally includes similar docs)
documents = [
    # Similar: Python basics
    "Python is a high-level programming language known for its clear syntax and readability.",
    "Python is an interpreted language with simple, readable syntax that's easy to learn.",
    "The Python programming language emphasizes code readability and clean syntax.",
    
    # Similar: Data science
    "Python is widely used in data science with libraries like pandas and numpy.",
    "Data scientists prefer Python for its rich ecosystem including pandas, numpy, and scikit-learn.",
    
    # Similar: Web development
    "Django and Flask are popular Python web frameworks for building web applications.",
    "Python web development commonly uses frameworks like Django or Flask.",
    
    # Unique topics
    "Python's GIL (Global Interpreter Lock) can limit multi-threaded performance.",
    "List comprehensions in Python provide a concise way to create lists.",
    "Python 3.10 introduced structural pattern matching with the match statement."
]

# Embed all documents
print("Embedding documents...")
doc_embeddings = get_embeddings_batch(documents)
print(f"Embedded {len(documents)} documents.")

In [ ]:
def standard_retrieve(query: str, top_k: int = 5) -> list[tuple[str, float]]:
    """Standard retrieval: return top-k by similarity."""
    query_embedding = get_embedding(query)
    
    # Score all documents
    scores = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    
    # Sort by score
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

# Test standard retrieval
query = "What is Python?"
print(f"Query: {query}\n")
print("=== Standard Retrieval (top-5) ===")
results = standard_retrieve(query, top_k=5)
for i, (doc, score) in enumerate(results, 1):
    print(f"{i}. [{score:.3f}] {doc}")

In [ ]:
def mmr_retrieve(
    query: str,
    top_k: int = 5,
    lambda_param: float = 0.5
) -> list[tuple[str, float, float]]:
    """
    MMR retrieval: balance relevance with diversity.
    
    Args:
        query: Search query
        top_k: Number of documents to return
        lambda_param: 0 = max diversity, 1 = max relevance
    
    Returns:
        List of (document, relevance_score, mmr_score) tuples
    """
    query_embedding = get_embedding(query)
    
    # Compute query-document similarities
    query_similarities = [
        cosine_similarity(query_embedding, doc_emb) 
        for doc_emb in doc_embeddings
    ]
    
    # Track selected documents
    selected_indices = []
    remaining_indices = list(range(len(documents)))
    results = []
    
    for _ in range(top_k):
        if not remaining_indices:
            break
        
        best_idx = None
        best_mmr = float('-inf')
        
        for idx in remaining_indices:
            # Relevance to query
            relevance = query_similarities[idx]
            
            # Max similarity to already selected documents
            if selected_indices:
                max_sim_to_selected = max(
                    cosine_similarity(doc_embeddings[idx], doc_embeddings[sel_idx])
                    for sel_idx in selected_indices
                )
            else:
                max_sim_to_selected = 0
            
            # MMR score
            mmr_score = lambda_param * relevance - (1 - lambda_param) * max_sim_to_selected
            
            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_idx = idx
        
        # Select the best document
        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)
        results.append((documents[best_idx], query_similarities[best_idx], best_mmr))
    
    return results

# Test MMR retrieval
print(f"Query: {query}\n")
print("=== MMR Retrieval (lambda=0.5, top-5) ===")
results = mmr_retrieve(query, top_k=5, lambda_param=0.5)
for i, (doc, rel_score, mmr_score) in enumerate(results, 1):
    print(f"{i}. [rel:{rel_score:.3f}, mmr:{mmr_score:.3f}] {doc}")

In [ ]:
# Compare different lambda values
print(f"Query: {query}\n")

for lambda_val in [1.0, 0.7, 0.5, 0.3, 0.0]:
    print(f"\n=== Lambda = {lambda_val} ===")
    if lambda_val == 1.0:
        print("(Pure relevance - same as standard retrieval)")
    elif lambda_val == 0.0:
        print("(Pure diversity - ignores relevance after first pick)")
    
    results = mmr_retrieve(query, top_k=3, lambda_param=lambda_val)
    for i, (doc, rel_score, mmr_score) in enumerate(results, 1):
        print(f"  {i}. {doc[:70]}...")

## Lambda Guidance

You should use higher lambda for precision focus and lower lambda for diversity focus.


## Part 2 Core Metrics

You will compute Precision, Recall, MRR, NDCG, and MAP.


In [ ]:
# Create evaluation dataset with ground truth
@dataclass
class EvalQuery:
    """A query with known relevant documents."""
    query: str
    relevant_doc_indices: list[int]  # Indices of relevant documents
    relevance_grades: dict[int, int] = field(default_factory=dict)  # For graded relevance (NDCG)

# Define evaluation queries with ground truth
eval_queries = [
    EvalQuery(
        query="What is Python and how readable is it?",
        relevant_doc_indices=[0, 1, 2],  # The three "basics" documents
        relevance_grades={0: 3, 1: 3, 2: 3, 3: 1, 4: 1}  # 3=highly relevant, 1=somewhat
    ),
    EvalQuery(
        query="Python for data science and analytics",
        relevant_doc_indices=[3, 4],  # Data science documents
        relevance_grades={3: 3, 4: 3, 0: 1, 1: 1, 2: 1}
    ),
    EvalQuery(
        query="Building web apps with Python",
        relevant_doc_indices=[5, 6],  # Web development documents
        relevance_grades={5: 3, 6: 3}
    ),
    EvalQuery(
        query="Python performance and threading",
        relevant_doc_indices=[7],  # GIL document
        relevance_grades={7: 3}
    ),
    EvalQuery(
        query="Python syntax features and list comprehensions",
        relevant_doc_indices=[8, 9, 0, 1, 2],  # Syntax-related
        relevance_grades={8: 3, 9: 2, 0: 2, 1: 2, 2: 2}
    )
]

print(f"Created {len(eval_queries)} evaluation queries with ground truth.")

In [ ]:
def precision_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """
    Precision@K: What fraction of top-K results are relevant?
    
    Precision@K = (# relevant in top-K) / K
    """
    top_k = retrieved_indices[:k]
    relevant_in_top_k = sum(1 for idx in top_k if idx in relevant_indices)
    return relevant_in_top_k / k

def recall_at_k(retrieved_indices: list[int], relevant_indices: list[int], k: int) -> float:
    """
    Recall@K: What fraction of relevant docs are in top-K?
    
    Recall@K = (# relevant in top-K) / (# total relevant)
    """
    if not relevant_indices:
        return 0.0
    top_k = retrieved_indices[:k]
    relevant_in_top_k = sum(1 for idx in top_k if idx in relevant_indices)
    return relevant_in_top_k / len(relevant_indices)

# Test on a single query
test_query = eval_queries[0]
print(f"Query: {test_query.query}")
print(f"Relevant docs: {test_query.relevant_doc_indices}")

# Get retrieval results
results = standard_retrieve(test_query.query, top_k=10)
retrieved_indices = [documents.index(doc) for doc, _ in results]
print(f"Retrieved order: {retrieved_indices}\n")

for k in [1, 3, 5, 10]:
    p = precision_at_k(retrieved_indices, test_query.relevant_doc_indices, k)
    r = recall_at_k(retrieved_indices, test_query.relevant_doc_indices, k)
    print(f"K={k}: Precision={p:.2f}, Recall={r:.2f}")

## Precision and Recall

You will measure quality and coverage at cutoff K.


In [ ]:
def reciprocal_rank(retrieved_indices: list[int], relevant_indices: list[int]) -> float:
    """
    Reciprocal Rank: 1 / (rank of first relevant result)
    
    If first relevant is at position 1: RR = 1.0
    If first relevant is at position 2: RR = 0.5
    If first relevant is at position 3: RR = 0.333
    If no relevant found: RR = 0.0
    """
    for i, idx in enumerate(retrieved_indices):
        if idx in relevant_indices:
            return 1.0 / (i + 1)
    return 0.0

def mean_reciprocal_rank(queries: list[EvalQuery], retrieval_fn) -> float:
    """
    MRR: Average reciprocal rank across queries.
    """
    rr_sum = 0.0
    for q in queries:
        results = retrieval_fn(q.query, top_k=10)
        retrieved_indices = [documents.index(doc) for doc, _ in results]
        rr_sum += reciprocal_rank(retrieved_indices, q.relevant_doc_indices)
    return rr_sum / len(queries)

# Calculate MRR
mrr = mean_reciprocal_rank(eval_queries, standard_retrieve)
print(f"Mean Reciprocal Rank (MRR): {mrr:.3f}")

# Show individual RRs
print("\nIndividual Reciprocal Ranks:")
for q in eval_queries:
    results = standard_retrieve(q.query, top_k=10)
    retrieved_indices = [documents.index(doc) for doc, _ in results]
    rr = reciprocal_rank(retrieved_indices, q.relevant_doc_indices)
    print(f"  RR={rr:.3f}: {q.query[:50]}...")

## MRR

You will measure how quickly users get the first useful result.


In [ ]:
def dcg_at_k(retrieved_indices: list[int], relevance_grades: dict[int, int], k: int) -> float:
    """
    Discounted Cumulative Gain at K.
    
    DCG@K = sum_{i=1}^{K} (2^{rel_i} - 1) / log2(i + 1)
    
    Higher relevance grades contribute more, but position matters (logarithmic discount).
    """
    dcg = 0.0
    for i, idx in enumerate(retrieved_indices[:k]):
        rel = relevance_grades.get(idx, 0)
        dcg += (2**rel - 1) / np.log2(i + 2)  # i+2 because log2(1) = 0
    return dcg

def ndcg_at_k(retrieved_indices: list[int], relevance_grades: dict[int, int], k: int) -> float:
    """
    Normalized DCG at K.
    
    NDCG@K = DCG@K / IDCG@K
    
    Where IDCG is the ideal DCG (perfect ranking).
    Returns 0 if no relevant documents exist.
    """
    # Actual DCG
    dcg = dcg_at_k(retrieved_indices, relevance_grades, k)
    
    # Ideal DCG: sort by relevance grade (descending)
    ideal_order = sorted(relevance_grades.keys(), key=lambda x: relevance_grades[x], reverse=True)
    idcg = dcg_at_k(ideal_order, relevance_grades, k)
    
    if idcg == 0:
        return 0.0
    return dcg / idcg

# Test NDCG
test_query = eval_queries[0]
print(f"Query: {test_query.query}")
print(f"Relevance grades: {test_query.relevance_grades}")

results = standard_retrieve(test_query.query, top_k=10)
retrieved_indices = [documents.index(doc) for doc, _ in results]
print(f"Retrieved order: {retrieved_indices}\n")

for k in [1, 3, 5, 10]:
    dcg = dcg_at_k(retrieved_indices, test_query.relevance_grades, k)
    ndcg = ndcg_at_k(retrieved_indices, test_query.relevance_grades, k)
    print(f"K={k}: DCG={dcg:.3f}, NDCG={ndcg:.3f}")

## DCG and NDCG

You will measure ranking quality with position awareness.


In [ ]:
def average_precision(retrieved_indices: list[int], relevant_indices: list[int]) -> float:
    """
    Average Precision: Average of precision values at each relevant document.
    
    AP = (1/|R|) * sum_{k: d_k is relevant} Precision@k
    
    Rewards retrieving relevant documents early.
    """
    if not relevant_indices:
        return 0.0
    
    precision_sum = 0.0
    relevant_found = 0
    
    for i, idx in enumerate(retrieved_indices):
        if idx in relevant_indices:
            relevant_found += 1
            precision_at_i = relevant_found / (i + 1)
            precision_sum += precision_at_i
    
    return precision_sum / len(relevant_indices)

def mean_average_precision(queries: list[EvalQuery], retrieval_fn) -> float:
    """
    MAP: Average of Average Precision across queries.
    """
    ap_sum = 0.0
    for q in queries:
        results = retrieval_fn(q.query, top_k=len(documents))
        retrieved_indices = [documents.index(doc) for doc, _ in results]
        ap_sum += average_precision(retrieved_indices, q.relevant_doc_indices)
    return ap_sum / len(queries)

# Calculate MAP
map_score = mean_average_precision(eval_queries, standard_retrieve)
print(f"Mean Average Precision (MAP): {map_score:.3f}")

# Show individual APs
print("\nIndividual Average Precisions:")
for q in eval_queries:
    results = standard_retrieve(q.query, top_k=len(documents))
    retrieved_indices = [documents.index(doc) for doc, _ in results]
    ap = average_precision(retrieved_indices, q.relevant_doc_indices)
    print(f"  AP={ap:.3f}: {q.query[:50]}...")

## Average Precision and MAP

You will measure full ranking quality across multiple queries.


## Part 3 Evaluation Harness

You will build reusable evaluation code for retrievers.


In [ ]:
@dataclass
class RetrievalMetrics:
    """Container for all retrieval metrics."""
    precision_at_1: float
    precision_at_3: float
    precision_at_5: float
    recall_at_1: float
    recall_at_3: float
    recall_at_5: float
    mrr: float
    ndcg_at_3: float
    ndcg_at_5: float
    map_score: float
    
    def __str__(self):
        return f"""
Retrieval Metrics:
  Precision:  @1={self.precision_at_1:.3f}  @3={self.precision_at_3:.3f}  @5={self.precision_at_5:.3f}
  Recall:     @1={self.recall_at_1:.3f}  @3={self.recall_at_3:.3f}  @5={self.recall_at_5:.3f}
  MRR:        {self.mrr:.3f}
  NDCG:       @3={self.ndcg_at_3:.3f}  @5={self.ndcg_at_5:.3f}
  MAP:        {self.map_score:.3f}
"""

def evaluate_retriever(
    queries: list[EvalQuery],
    retrieval_fn,
    k_values: list[int] = [1, 3, 5]
) -> RetrievalMetrics:
    """Compute all metrics for a retrieval function."""
    
    # Collect per-query metrics
    all_results = []
    for q in queries:
        results = retrieval_fn(q.query, top_k=max(k_values) + 5)
        retrieved_indices = [documents.index(doc) for doc, *_ in results]
        all_results.append((q, retrieved_indices))
    
    # Compute averages
    def avg(fn, k=None):
        if k:
            return np.mean([fn(r, q.relevant_doc_indices, k) for q, r in all_results])
        return np.mean([fn(r, q.relevant_doc_indices) for q, r in all_results])
    
    def avg_ndcg(k):
        return np.mean([ndcg_at_k(r, q.relevance_grades, k) for q, r in all_results])
    
    return RetrievalMetrics(
        precision_at_1=avg(precision_at_k, 1),
        precision_at_3=avg(precision_at_k, 3),
        precision_at_5=avg(precision_at_k, 5),
        recall_at_1=avg(recall_at_k, 1),
        recall_at_3=avg(recall_at_k, 3),
        recall_at_5=avg(recall_at_k, 5),
        mrr=avg(reciprocal_rank),
        ndcg_at_3=avg_ndcg(3),
        ndcg_at_5=avg_ndcg(5),
        map_score=np.mean([average_precision(r, q.relevant_doc_indices) for q, r in all_results])
    )

# Evaluate standard retrieval
print("=== Standard Retrieval ===")
metrics = evaluate_retriever(eval_queries, standard_retrieve)
print(metrics)

In [ ]:
# Create MMR retrieval functions with different lambdas
def mmr_retrieve_07(query: str, top_k: int = 5):
    return mmr_retrieve(query, top_k, lambda_param=0.7)

def mmr_retrieve_05(query: str, top_k: int = 5):
    return mmr_retrieve(query, top_k, lambda_param=0.5)

def mmr_retrieve_03(query: str, top_k: int = 5):
    return mmr_retrieve(query, top_k, lambda_param=0.3)

# Compare all retrievers
retrievers = [
    ("Standard (λ=1.0)", standard_retrieve),
    ("MMR (λ=0.7)", mmr_retrieve_07),
    ("MMR (λ=0.5)", mmr_retrieve_05),
    ("MMR (λ=0.3)", mmr_retrieve_03),
]

print("Comparing Retrievers:")
print("=" * 70)

for name, fn in retrievers:
    metrics = evaluate_retriever(eval_queries, fn)
    print(f"\n{name}:")
    print(f"  P@3: {metrics.precision_at_3:.3f}  R@3: {metrics.recall_at_3:.3f}  MRR: {metrics.mrr:.3f}  MAP: {metrics.map_score:.3f}")

## Metric Reading

You will read metrics together to diagnose behavior.


## Part 4 Advanced MMR

You will add adaptive retrieval behavior.


In [ ]:
def adaptive_mmr_retrieve(
    query: str,
    top_k: int = 5,
    specificity_threshold: float = 0.7
) -> list[tuple[str, float, float]]:
    """
    Adaptive MMR: Adjust lambda based on query specificity.
    
    Specific queries (high max similarity to any doc) -> high lambda
    Broad queries (moderate similarity to many docs) -> low lambda
    """
    query_embedding = get_embedding(query)
    
    # Compute max similarity to any document
    similarities = [cosine_similarity(query_embedding, doc_emb) for doc_emb in doc_embeddings]
    max_sim = max(similarities)
    std_sim = np.std(similarities)
    
    # High max + low std = specific query (use high lambda)
    # Moderate max + high std = broad query (use low lambda)
    if max_sim > specificity_threshold and std_sim > 0.05:
        lambda_param = 0.8  # Specific query, prioritize relevance
    elif max_sim < specificity_threshold - 0.1:
        lambda_param = 0.4  # Vague query, prioritize diversity
    else:
        lambda_param = 0.6  # Balanced
    
    print(f"  [Adaptive: max_sim={max_sim:.2f}, std={std_sim:.2f} -> λ={lambda_param}]")
    return mmr_retrieve(query, top_k, lambda_param)

# Test adaptive MMR
test_queries = [
    "What is Python?",  # Broad
    "Python GIL threading performance",  # Specific
    "Python web frameworks Django Flask",  # Moderately specific
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = adaptive_mmr_retrieve(q, top_k=3)
    for i, (doc, rel, mmr) in enumerate(results, 1):
        print(f"  {i}. {doc[:60]}...")

## Exercises With Starter and Solution

### Exercise 1

You will add document importance weights to MMR.


In [ ]:
# Starter code for Exercise 1
def weighted_mmr_retrieve(
    query: str,
    doc_weights: list[float],
    top_k: int = 5,
    lambda_param: float = 0.5,
    weight_factor: float = 0.2
) -> list[tuple[str, float, float]]:
    # TODO combine relevance, diversity, and importance weights
    pass


In [ ]:
# Solution for Exercise 1
def weighted_mmr_retrieve(
    query: str,
    doc_weights: list[float],
    top_k: int = 5,
    lambda_param: float = 0.5,
    weight_factor: float = 0.2
) -> list[tuple[str, float, float]]:
    if len(doc_weights) != len(documents):
        raise ValueError('doc_weights must match document count')

    query_embedding = get_embedding(query)
    query_similarities = [cosine_similarity(query_embedding, emb) for emb in doc_embeddings]

    selected = []
    remaining = list(range(len(documents)))
    results = []

    for _ in range(top_k):
        if not remaining:
            break

        best_idx = None
        best_score = float('-inf')

        for idx in remaining:
            relevance = query_similarities[idx] + weight_factor * doc_weights[idx]

            if selected:
                max_sim_selected = max(cosine_similarity(doc_embeddings[idx], doc_embeddings[s]) for s in selected)
            else:
                max_sim_selected = 0.0

            mmr_score = lambda_param * relevance - (1 - lambda_param) * max_sim_selected

            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)
        results.append((documents[best_idx], query_similarities[best_idx], best_score))

    return results

weights = [1.0] * len(documents)
weights[7] = 1.3
for i, row in enumerate(weighted_mmr_retrieve('Python performance', weights, top_k=3), 1):
    print(i, row[0][:65], round(row[2], 3))


### Exercise 2

You will implement a diversity aware precision metric.


In [ ]:
# Starter code for Exercise 2
def diversity_penalized_precision(
    retrieved_indices: list[int],
    relevant_indices: list[int],
    k: int,
    redundancy_penalty: float = 0.1
) -> float:
    # TODO penalize redundant top K result sets
    pass


In [ ]:
# Solution for Exercise 2
def diversity_penalized_precision(
    retrieved_indices: list[int],
    relevant_indices: list[int],
    k: int,
    redundancy_penalty: float = 0.1
) -> float:
    top_k = retrieved_indices[:k]
    if not top_k:
        return 0.0

    base_precision = sum(1 for idx in top_k if idx in relevant_indices) / k

    if len(top_k) < 2:
        return base_precision

    pair_sims = []
    for i in range(len(top_k)):
        for j in range(i + 1, len(top_k)):
            pair_sims.append(cosine_similarity(doc_embeddings[top_k[i]], doc_embeddings[top_k[j]]))

    avg_redundancy = float(np.mean(pair_sims)) if pair_sims else 0.0
    penalty = redundancy_penalty * max(0.0, avg_redundancy - 0.6)

    return max(0.0, min(1.0, base_precision - penalty))

print(diversity_penalized_precision([0, 1, 2, 7, 9], [0, 2, 7], k=5))


### Exercise 3

You will run a bootstrap style A B retriever comparison.


In [ ]:
# Starter code for Exercise 3
def ab_test_retrievers(
    queries: list[EvalQuery],
    retriever_a,
    retriever_b,
    metric_fn,
    n_bootstrap: int = 1000
) -> dict:
    # TODO compare retrievers and return confidence interval
    pass


In [ ]:
# Solution for Exercise 3
def ab_test_retrievers(
    queries: list[EvalQuery],
    retriever_a,
    retriever_b,
    metric_fn,
    n_bootstrap: int = 1000
) -> dict:
    def call_metric(retrieved, relevant):
        try:
            return float(metric_fn(retrieved, relevant))
        except TypeError:
            return float(metric_fn(retrieved, relevant, 5))

    scores_a = []
    scores_b = []

    for q in queries:
        ra = retriever_a(q.query, top_k=len(documents))
        rb = retriever_b(q.query, top_k=len(documents))

        idx_a = [documents.index(doc) for doc, *_ in ra]
        idx_b = [documents.index(doc) for doc, *_ in rb]

        scores_a.append(call_metric(idx_a, q.relevant_doc_indices))
        scores_b.append(call_metric(idx_b, q.relevant_doc_indices))

    scores_a = np.array(scores_a)
    scores_b = np.array(scores_b)
    diff = scores_b - scores_a
    observed = float(np.mean(diff))

    rng = np.random.default_rng(42)
    boot = []
    n = len(queries)
    for _ in range(n_bootstrap):
        sample = rng.integers(0, n, size=n)
        boot.append(float(np.mean(diff[sample])))

    boot = np.array(boot)
    ci_low = float(np.percentile(boot, 2.5))
    ci_high = float(np.percentile(boot, 97.5))
    p_value = float(2 * min(np.mean(boot <= 0), np.mean(boot >= 0)))

    return {
        'mean_a': float(np.mean(scores_a)),
        'mean_b': float(np.mean(scores_b)),
        'observed_diff_b_minus_a': observed,
        'ci_95': [ci_low, ci_high],
        'p_value': p_value,
        'winner': 'B' if observed > 0 else 'A',
    }

mmr_fn = lambda query, top_k=10: mmr_retrieve(query, top_k=top_k, lambda_param=0.5)
metric = lambda retrieved, relevant: precision_at_k(retrieved, relevant, 3)
print(ab_test_retrievers(eval_queries, standard_retrieve, mmr_fn, metric, n_bootstrap=400))


## What You Learned

You can now improve retrieval quality with diversity controls and objective evaluation.
